# 👁️ Can AI Be a Doctor? — Eye Disease Detection with Vision-Language Models

## Welcome to this AI Ophthalmology Lab!

In this notebook, we're going to explore a cutting-edge research question:

> **"Can a large AI model — trained on images and text from the internet — diagnose eye diseases from medical scans?"**

This is based on the paper **LMOD: A Large Multimodal Ophthalmology Dataset** (Qin et al., 2025), which benchmarked 13 state-of-the-art AI models on real ophthalmology data.

---

### 🎯 What will we do?

We'll run **three real tasks** that doctors perform every day:

| Task | What it means | Image type |
|------|--------------|------------|
| 🔬 **Anatomical Recognition** | Find and label structures in a retinal scan | OCT (cross-section of the eye) |
| 👁️ **Glaucoma Diagnosis** | Decide if a photo of the retina shows glaucoma | Color Fundus Photo |
| 🕳️ **Macular Hole Staging** | Rate the severity (1–4) of a hole in the retina | OCT |

We'll use **InternVL3.5-8B**, a powerful open-source vision-language model.

---

### 🔑 Key vocabulary
- **OCT (Optical Coherence Tomography)**: A special camera that takes a cross-section "slice" of the eye, like an MRI but for the retina.
- **Fundus photo**: A photo of the back of the eye (the retina), taken through the pupil.
- **Glaucoma**: An eye disease where the optic nerve is damaged, often leading to blindness.
- **Macular hole**: A small hole in the centre of the retina that causes blurry or missing central vision.
- **LVLM (Large Vision-Language Model)**: An AI that can both look at images and read/write text — like ChatGPT but with eyes!

---
**⚠️ Runtime tip**: This notebook requires a GPU. InternVL3.5-8B needs ~16 GB of GPU RAM — an **A100 GPU (Colab Pro)** is recommended. Go to **Runtime → Change runtime type → A100 GPU**.

## Step 1: Install Required Packages

We need to install some Python libraries. This might take a couple of minutes.

In [ ]:
%%capture
# Pin transformers to <4.46 — newer versions break InternVL2's custom model code
!pip install "transformers>=4.37.0,<4.46.0" accelerate einops timm sentencepiece
!pip install flash-attn --no-build-isolation  # Speeds up the model (optional but recommended)
print("✅ Packages installed!")

In [ ]:
import os
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f"✅ GPU found: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — the model will be very slow. Please enable GPU in Runtime settings.")

## Step 2: Clone the Sample Repository

All sample images are stored in the [MaJinWakeUp/MedVLM](https://github.com/MaJinWakeUp/MedVLM) GitHub repository alongside this notebook. The cell below clones it automatically — **no file uploads or Google Drive needed**.

Just run the cell below — done!

In [ ]:
GITHUB_REPO = "MaJinWakeUp/MedVLM"

!git clone --depth 1 https://github.com/{GITHUB_REPO}.git /content/MedVLM
print("✅ Repository cloned to /content/MedVLM")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA PATHS — samples/ is inside the cloned repo at /content/MedVLM/
# ─────────────────────────────────────────────────────────────────────────────

SAMPLES_DIR = '/content/MedVLM/samples'
OIMHS_DIR   = os.path.join(SAMPLES_DIR, 'OIMHS')
REFUGE_DIR  = os.path.join(SAMPLES_DIR, 'REFUGE')

for label, path in [('samples/',        SAMPLES_DIR),
                    ('samples/OIMHS/',   OIMHS_DIR),
                    ('samples/REFUGE/', REFUGE_DIR)]:
    ok = os.path.isdir(path)
    print(f"  {'✅' if ok else '❌'} {label:20s} → {path}")

if not os.path.isdir(SAMPLES_DIR):
    print("\n⚠️  'samples/' not found. Check that the git clone above completed without errors.")

## Step 3: Load Our 10 Sample Images

We have **5 OCT samples** (one from each macular hole stage, stages 1–4) and **5 fundus photos** (2 with glaucoma, 3 without). Let's load their metadata.

In [ ]:
# ─── Load 5 OCT samples ───────────────────────────────────────────────────────
# Each entry: (sample_id, macular_hole_stage)
OCT_SAMPLE_IDS = [
    ('35_8',   1),   # Stage 1 — early pit
    ('28_13',  2),   # Stage 2 — small hole
    ('102_32', 3),   # Stage 3 — full hole, vitreous attached
    ('100_17', 4),   # Stage 4 — full hole, vitreous detached
    ('100_18', 4),   # Stage 4 — another example
]

oct_samples = []
for sample_id, expected_stage in OCT_SAMPLE_IDS:
    info_path = os.path.join(OIMHS_DIR, sample_id, 'information.json')
    with open(info_path) as f:
        meta = json.load(f)
    stage = meta['metadata']['stage_of_macular_hole_decision']
    regions = {bb['annotation_ID']: bb['region_type']
               for bb in meta['annotations']['bounding_boxes']}
    oct_samples.append({
        'id': sample_id,
        'meta': meta,
        'stage': stage,
        'task_type': 'oct'
    })
    print(f"  OCT  [{sample_id:8s}] MH Stage {stage} | Regions: {', '.join(set(regions.values()))}")

print()

# ─── Load 5 Fundus samples ────────────────────────────────────────────────────
# 3 Non-Glaucoma, 2 Glaucoma
CFP_SAMPLE_IDS = [
    'V0001',   # Non-Glaucoma
    'V0002',   # Non-Glaucoma
    'V0003',   # Non-Glaucoma
    'V0006',   # Glaucoma
    'V0026',   # Glaucoma
]

cfp_samples = []
for sample_id in CFP_SAMPLE_IDS:
    info_path = os.path.join(REFUGE_DIR, sample_id, 'information.json')
    with open(info_path) as f:
        meta = json.load(f)
    label = meta['metadata']['glaucoma_label']
    cfp_samples.append({
        'id': sample_id,
        'meta': meta,
        'label': label,
        'task_type': 'cfp'
    })
    print(f"  CFP  [{sample_id:8s}] Ground Truth: {label}")

print(f"\nTotal: {len(oct_samples)} OCT + {len(cfp_samples)} Fundus = {len(oct_samples)+len(cfp_samples)} samples ✅")

## Step 4: Visualise the Sample Images

Before asking the AI, let's look at our images. The **annotated** versions show the bounding boxes that doctors drew — these are what we'll pass to the AI.

In [ ]:
def load_oct_image(sample, annotated=True):
    """Load an OCT image from the samples directory."""
    base = os.path.join(OIMHS_DIR, sample['id'])
    path = os.path.join(base, 'annotated', 'bbox_annotated.png') if annotated \
           else os.path.join(base, 'visualization.png')
    return Image.open(path).convert('RGB')

def load_cfp_image(sample, annotated=True):
    """Load a fundus photo from the samples directory."""
    base = os.path.join(REFUGE_DIR, sample['id'])
    path = os.path.join(base, 'annotated', 'annotated_bounding_box.png') if annotated \
           else os.path.join(base, 'visualization.png')
    return Image.open(path).convert('RGB')


# ─── Plot OCT samples ────────────────────────────────────────────────────────
stage_descriptions = {
    1: 'Stage 1\n(Early — small pit)',
    2: 'Stage 2\n(Small hole forms)',
    3: 'Stage 3\n(Full hole,\nvitreous attached)',
    4: 'Stage 4\n(Full hole,\nvitreous detached)'
}

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('OCT Scans — Cross-sections of the Retina (with expert annotations)',
             fontsize=14, fontweight='bold', y=1.02)

for ax, sample in zip(axes, oct_samples):
    ax.imshow(load_oct_image(sample, annotated=True))
    ax.set_title(f"Sample: {sample['id']}\n{stage_descriptions[sample['stage']]}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('oct_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print("💡 Coloured boxes drawn by ophthalmology experts mark:")
print("   irc = IntraRetinal Cyst  |  retina = Retina  |  choroid = Choroid  |  mh = Macular Hole")

In [ ]:
# ─── Plot Fundus Photo samples ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle('Color Fundus Photographs — Photos of the Back of the Eye',
             fontsize=14, fontweight='bold', y=1.02)

label_colors = {'Glaucoma': 'red', 'Non-Glaucoma': 'green'}

for ax, sample in zip(axes, cfp_samples):
    ax.imshow(load_cfp_image(sample, annotated=True))
    label = sample['label']
    ax.set_title(f"Sample: {sample['id']}", fontsize=9)
    ax.text(0.5, -0.05, f"GT: {label}", transform=ax.transAxes,
            ha='center', fontsize=9, color=label_colors[label], fontweight='bold')
    # Hide ticks but keep spine visible so the colored border shows
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(label_colors[label])
        spine.set_linewidth(5)

plt.tight_layout()
plt.savefig('cfp_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print("Red border = Glaucoma  |  Green border = Non-Glaucoma")
print("The annotated boxes highlight the optic disc and cup (key for glaucoma diagnosis)")

## Step 5: Load InternVL3.5-8B — Our AI Model

### 🤖 What is InternVL3.5?

InternVL3.5 is a **Large Vision-Language Model (LVLM)** — it can look at images *and* understand text, combining the powers of:
- A **vision encoder** (like a brain for seeing) — based on the same architecture used in CLIP
- A **language model** (like a brain for reading/writing) — similar to how ChatGPT works

The InternVL model family was the **top-performing open-source series** in the LMOD paper:
- InternVL-4B achieved F1 of **0.55** in anatomical recognition (average across all models was only 0.22)
- InternVL-2B got **30.3%** accuracy in macular hole staging (best among open-source models)

We'll use **InternVL3.5-8B** (8 billion parameters) — a larger, more capable model than the one tested in the paper. Let's see if it does better!

---
**Memory note:** 8B parameters in bfloat16 requires ~16 GB GPU RAM. Use an **A100 GPU** in Colab (Runtime → Change runtime type → A100). On a T4, the model will load slowly using CPU offloading.

*This downloads ~16 GB from HuggingFace on the first run. Subsequent runs use the cached version.*

In [ ]:
# Image preprocessing — InternVL expects images resized and normalised to ImageNet stats
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def preprocess_image(pil_image, input_size=448):
    """Resize and normalise a PIL image for InternVL input."""
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return transform(pil_image).unsqueeze(0).to(torch.bfloat16).to(device)


MODEL_NAME = 'OpenGVLab/InternVL3_5-8B'
print(f"Loading {MODEL_NAME} from HuggingFace (~16 GB download on first run)...")
print("Tip: A100 GPU recommended. On T4, the model will use CPU offloading (slower).\n")

try:
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        device_map='auto',          # Spreads layers across GPU/CPU to fit available memory
        use_flash_attn=True,        # Faster attention (requires flash-attn package)
        trust_remote_code=True,     # InternVL uses custom model code
    ).eval()
except Exception:
    print("Flash attention not available, loading without it...")
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        device_map='auto',
        use_flash_attn=False,
        trust_remote_code=True,
    ).eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=False)

# Deterministic generation (temperature=0 equivalent), max 512 output tokens
generation_config = dict(max_new_tokens=512, do_sample=False)

print(f"\n✅ Model loaded! Parameters: {sum(p.numel() for p in model.parameters())/1e9:.1f}B")
print(f"   Running on: {device} (with device_map='auto')")

## Task 1: Anatomical Recognition in OCT Images

### 🔬 The Challenge

An OCT scan shows layers of the retina in cross-section. We need the AI to look at an image with **numbered bounding boxes** (coloured rectangles) drawn on it and tell us **what each box is** — just like a doctor would.

Possible region types:
- **irc** — IntraRetinal Cyst (fluid-filled pockets — a sign of disease)
- **retina** — The main light-sensing layer at the back of the eye
- **choroid** — The blood-supply layer behind the retina
- **mh** — Macular Hole

### 📝 The Prompt
We ask the model:
> *"This is an ophthalmology OCT image. Please identify the type of each labeled bounding box. Options can be: irc, retina, choroid, mh. Follow the format: Region ID: X; Type: Y"*

In [ ]:
def build_anatomical_prompt(bounding_boxes):
    """Build the prompt listing available region IDs and possible types."""
    region_ids = [str(bb['annotation_ID']) for bb in bounding_boxes]
    return (
        "This is an ophthalmology OCT image. "
        "Please identify the type of each labeled bounding box in this image. "
        "Options can be: irc, retina, choroid, mh. "
        "Please just follow the format: "
        + "; ".join([f"Region ID: {rid}; Type: <answer>" for rid in region_ids])
    )

def parse_anatomical_response(response):
    """Parse model's response into a dict of {region_id: predicted_type}.

    Handles responses like: "Region ID: 1; Type: retina; Region ID: 2; Type: choroid"
    where each ID and type are in adjacent semicolon-separated segments.
    """
    predictions = {}
    # Match "Region ID: X" immediately followed (after separator) by "Type: Y"
    pairs = re.findall(
        r'region\s+id[:\s]+(\d+)[^a-z]*type[:\s]+([a-z]+)',
        response, re.IGNORECASE
    )
    for rid, rtype in pairs:
        predictions[rid] = rtype.lower().rstrip('.,;')
    return predictions


print("Running Task 1: Anatomical Recognition on 5 OCT images...\n")
task1_results = []

for i, sample in enumerate(oct_samples):
    img          = load_oct_image(sample, annotated=True)
    pixel_values = preprocess_image(img)
    bboxes       = sample['meta']['annotations']['bounding_boxes']
    gt           = {str(bb['annotation_ID']): bb['region_type'] for bb in bboxes}
    prompt       = f'<image>\n{build_anatomical_prompt(bboxes)}'

    with torch.no_grad():
        response = model.chat(tokenizer, pixel_values, prompt, generation_config)

    predictions = parse_anatomical_response(response)
    correct  = sum(predictions.get(rid, '') == rtype for rid, rtype in gt.items())
    total    = len(gt)
    accuracy = correct / total if total > 0 else 0

    task1_results.append({
        'sample': sample, 'image': img,
        'ground_truth': gt, 'predictions': predictions,
        'response': response,
        'accuracy': accuracy, 'correct': correct, 'total': total
    })

    status = '✅' if accuracy == 1.0 else ('⚠️' if accuracy > 0 else '❌')
    print(f"Sample {i+1} [{sample['id']}] Stage {sample['stage']}: {correct}/{total} correct {status}")
    print(f"  GT:       {gt}")
    print(f"  Pred:     {predictions}")
    print(f"  Response: {response[:120].strip()}...")
    print()

In [ ]:
# ─── Visualise Task 1 Results ────────────────────────────────────────────────
region_name_map = {
    'irc': 'IntraRetinal Cyst', 'retina': 'Retina',
    'choroid': 'Choroid',       'mh': 'Macular Hole',
}

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle('Task 1: Anatomical Recognition — Model vs. Expert',
             fontsize=15, fontweight='bold', y=1.01)

for col, result in enumerate(task1_results):
    axes[0][col].imshow(result['image'])
    axes[0][col].set_title(
        f"Sample {col+1}: {result['sample']['id']}\nMH Stage: {result['sample']['stage']}",
        fontsize=9)
    axes[0][col].axis('off')

    axes[1][col].axis('off')
    gt, pred = result['ground_truth'], result['predictions']
    text_lines = [f"Score: {result['correct']}/{result['total']}\n"]
    for rid, true_type in sorted(gt.items()):
        p_type = pred.get(rid, '?')
        icon = '[OK]' if p_type == true_type else '[ X]'
        text_lines += [
            f"{icon} Box {rid}:",
            f"  GT:  {region_name_map.get(true_type, true_type)}",
            f"  AI:  {region_name_map.get(p_type, p_type)}", ""
        ]
    bg = '#d4edda' if result['accuracy'] == 1.0 else ('#fff3cd' if result['accuracy'] > 0 else '#f8d7da')
    axes[1][col].text(0.05, 0.95, '\n'.join(text_lines),
                      transform=axes[1][col].transAxes, va='top', fontsize=8,
                      fontfamily='monospace',
                      bbox=dict(boxstyle='round', facecolor=bg, alpha=0.8))

plt.tight_layout()
plt.savefig('task1_results.png', dpi=100, bbox_inches='tight')
plt.show()

overall_acc = np.mean([r['accuracy'] for r in task1_results])
print(f"\n📊 Overall Task 1 Accuracy: {overall_acc:.1%}")
print(f"   Paper's InternVL-2B F1 on full OCT dataset: 0.4807")

## Task 2: Glaucoma Diagnosis

### 👁️ The Challenge

**Glaucoma** is the leading cause of irreversible blindness worldwide. It's diagnosed by examining the **optic disc** (the bright circle where the nerve leaves the eye) — specifically the ratio between the "cup" (inner circle) and "disc" (outer circle).

- **High cup-to-disc ratio** → More likely to have glaucoma
- The annotated images show these regions in coloured boxes

### 📝 The Prompt
> *"This is a color fundus image. Based on the image, please tell me whether this image contains glaucoma. Follow the format: GLAUCOMA / NON-GLAUCOMA; Explanation: \<JUSTIFICATION\>"*

In [ ]:
GLAUCOMA_PROMPT = (
    "This is a color fundus image of type Fundus RGB Images. "
    "Based on the image, please tell me whether this image contains glaucoma, "
    "then give detailed justifications. "
    "Follow the format: GLAUCOMA / NON-GLAUCOMA; Explanation: <JUSTIFICATION>."
)

def parse_glaucoma_response(response):
    """Extract the GLAUCOMA or NON-GLAUCOMA decision from the model's response."""
    r = response.upper()
    if 'NON-GLAUCOMA' in r or 'NON GLAUCOMA' in r:
        return 'Non-Glaucoma'
    elif 'GLAUCOMA' in r:
        return 'Glaucoma'
    return 'Unknown'


print("Running Task 2: Glaucoma Diagnosis on 5 Fundus Photos...\n")
task2_results = []

for i, sample in enumerate(cfp_samples):
    img          = load_cfp_image(sample, annotated=True)
    pixel_values = preprocess_image(img)
    prompt       = f'<image>\n{GLAUCOMA_PROMPT}'

    with torch.no_grad():
        response = model.chat(tokenizer, pixel_values, prompt, generation_config)

    prediction = parse_glaucoma_response(response)
    gt_label   = sample['label']
    correct    = prediction == gt_label

    task2_results.append({
        'sample': sample, 'image': img,
        'ground_truth': gt_label, 'prediction': prediction,
        'response': response, 'correct': correct
    })

    icon = '✅' if correct else '❌'
    print(f"Sample {i+1} [{sample['id']}]: GT={gt_label:13s} | AI={prediction:13s} {icon}")
    print(f"  Reasoning: {response[:180].strip()}...")
    print()

In [ ]:
# ─── Visualise Task 2 Results ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
fig.suptitle('Task 2: Glaucoma Diagnosis — Model vs. Expert Label',
             fontsize=15, fontweight='bold', y=1.01)

for col, result in enumerate(task2_results):
    ax_img = axes[0][col]
    ax_img.imshow(result['image'])
    border_color = '#2ecc71' if result['correct'] else '#e74c3c'
    for spine in ax_img.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(5)
        spine.set_visible(True)
    ax_img.set_xticks([])
    ax_img.set_yticks([])
    ax_img.set_title(f"Sample {col+1}: {result['sample']['id']}", fontsize=9)

    ax_text = axes[1][col]
    ax_text.axis('off')
    gt, pred = result['ground_truth'], result['prediction']
    # Use plain text instead of emoji glyphs (matplotlib font doesn't support them)
    verdict    = 'CORRECT!' if result['correct'] else 'WRONG!'
    gt_color   = 'darkred'   if gt   == 'Glaucoma' else 'darkgreen'
    pred_color = 'darkred'   if pred == 'Glaucoma' else 'darkgreen'
    reasoning  = result['response']
    if 'Explanation:' in reasoning:
        reasoning = reasoning.split('Explanation:')[1].strip()
    reasoning = (reasoning[:200] + '...') if len(reasoning) > 200 else reasoning

    ax_text.text(0.5, 0.95, verdict,           transform=ax_text.transAxes, ha='center', va='top',
                 fontsize=11, fontweight='bold', color='green' if result['correct'] else 'red')
    ax_text.text(0.5, 0.80, f"Expert: {gt}", transform=ax_text.transAxes, ha='center', va='top',
                 fontsize=9, color=gt_color, fontweight='bold')
    ax_text.text(0.5, 0.68, f"AI says: {pred}", transform=ax_text.transAxes, ha='center', va='top',
                 fontsize=9, color=pred_color, fontweight='bold')
    ax_text.text(0.5, 0.55, 'AI Reasoning:', transform=ax_text.transAxes,
                 ha='center', va='top', fontsize=8, fontstyle='italic')
    ax_text.text(0.05, 0.45, reasoning, transform=ax_text.transAxes,
                 ha='left', va='top', fontsize=7, wrap=True,
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
plt.savefig('task2_results.png', dpi=100, bbox_inches='tight')
plt.show()

task2_acc = np.mean([r['correct'] for r in task2_results])
print(f"\n📊 Task 2 Accuracy on our 5 samples: {task2_acc:.1%}")
print(f"   Paper result for InternVL-2B on full dataset: 50% (close to random!)")
print(f"   Random guessing baseline: 50%")
print(f"   A trained specialist AI: ~82% (from paper, Table 4)")

## Task 3: Macular Hole Staging

### 🕳️ The Challenge

A **macular hole** is a small hole that forms in the macula — the central part of the retina responsible for sharp central vision. It can be graded from Stage 1 (early) to Stage 4 (severe):

| Stage | What's happening |
|-------|------------------|
| **1** | A small pit forms in the centre of the retina (fovea) |
| **2** | A small full-thickness hole appears (< 400 µm) |
| **3** | A larger full-thickness hole (> 400 µm), vitreous still attached |
| **4** | A large hole, vitreous has fully detached |

This is harder than glaucoma — a 4-way classification!

### 📝 The Prompt
> *"This is an ophthalmology OCT image. Please tell me the stage of the macular hole. Follow the format: Stage: \<AN INTEGER\>; Justification: \<EXPLANATION\>"*

In [ ]:
MH_STAGING_PROMPT = (
    "This is an ophthalmology OCT image. "
    "Based on the image, please tell me the stage of macular hole decision. "
    "Then, give a detailed justification and explanation for your answer. "
    "Follow the format: Stage: <AN INTEGER>; Justification: <EXPLANATION>."
)

def parse_mh_stage_response(response):
    """Extract the stage number (1-4) from the model's response."""
    import re
    match = re.search(r'stage[:\s]+([1-4])', response, re.IGNORECASE)
    if match:
        return int(match.group(1))
    match = re.search(r'\b([1-4])\b', response)
    return int(match.group(1)) if match else None


print("Running Task 3: Macular Hole Staging on 5 OCT images...\n")
task3_results = []

for i, sample in enumerate(oct_samples):
    # Use the clean image (no bounding boxes) — we're asking about overall disease stage
    img          = load_oct_image(sample, annotated=False)
    pixel_values = preprocess_image(img)
    prompt       = f'<image>\n{MH_STAGING_PROMPT}'

    with torch.no_grad():
        response = model.chat(tokenizer, pixel_values, prompt, generation_config)

    gt_stage   = sample['stage']
    pred_stage = parse_mh_stage_response(response)
    correct    = pred_stage == gt_stage

    task3_results.append({
        'sample': sample, 'image': img,
        'ground_truth': gt_stage, 'prediction': pred_stage,
        'response': response, 'correct': correct
    })

    icon = '✅' if correct else '❌'
    print(f"Sample {i+1} [{sample['id']}]: GT=Stage {gt_stage} | AI=Stage {pred_stage} {icon}")
    print(f"  Response: {response[:200].strip()}...")
    print()

In [ ]:
# ─── Visualise Task 3 Results ────────────────────────────────────────────────
stage_colors = {1: '#3498db', 2: '#f39c12', 3: '#e67e22', 4: '#e74c3c'}
stage_labels = {1: 'Stage 1\nEarly Pit', 2: 'Stage 2\nSmall Hole',
                3: 'Stage 3\nFull Hole', 4: 'Stage 4\nDetached'}

fig, axes = plt.subplots(3, 5, figsize=(22, 14))
fig.suptitle('Task 3: Macular Hole Staging — Model vs. Expert Diagnosis',
             fontsize=15, fontweight='bold', y=1.01)

for col, result in enumerate(task3_results):
    gt, pred, correct = result['ground_truth'], result['prediction'], result['correct']

    # Row 0: OCT image
    axes[0][col].imshow(result['image'])
    axes[0][col].set_title(f"Sample {col+1}: {result['sample']['id']}", fontsize=9)
    axes[0][col].axis('off')

    # Row 1: Stage comparison bar chart
    ax_bar = axes[1][col]
    ax_bar.set_xlim(0, 1)
    ax_bar.set_ylim(0, 4.5)
    ax_bar.axis('off')
    ax_bar.set_title('Stage Comparison', fontsize=8)
    for stage in [1, 2, 3, 4]:
        y     = 4.5 - stage
        alpha = 1.0 if stage == gt else 0.15
        ax_bar.barh(y, 0.4, left=0.05, color=stage_colors[stage], alpha=alpha, height=0.7)
        ax_bar.text(0.5, y, stage_labels[stage].split('\n')[0], ha='left', va='center', fontsize=7)
        if stage == gt:
            ax_bar.text(0.03, y, 'GT', ha='left', va='center', fontsize=7, fontweight='bold')
        if stage == pred:
            # Plain text label — avoid emoji glyphs that matplotlib can't render
            label_color = 'green' if correct else 'red'
            label_text  = '[AI OK]' if correct else '[AI X]'
            ax_bar.text(0.03, y - 0.3, label_text,
                        ha='left', va='center', fontsize=7, color=label_color, fontweight='bold')

    # Row 2: AI reasoning text
    ax_text = axes[2][col]
    ax_text.axis('off')
    verdict   = 'Correct!' if correct else 'Wrong!'
    reasoning = result['response']
    if 'Justification:' in reasoning:
        reasoning = reasoning.split('Justification:')[1].strip()
    reasoning = (reasoning[:250] + '...') if len(reasoning) > 250 else reasoning
    bg = '#d4edda' if correct else '#f8d7da'
    ax_text.text(0.5, 1.0,  verdict,              transform=ax_text.transAxes, ha='center', va='top',
                 fontsize=10, fontweight='bold', color='green' if correct else 'red')
    ax_text.text(0.5, 0.88, f"GT: Stage {gt} | AI: Stage {pred}",
                 transform=ax_text.transAxes, ha='center', va='top', fontsize=8)
    ax_text.text(0.05, 0.75, reasoning, transform=ax_text.transAxes,
                 ha='left', va='top', fontsize=6.5, wrap=True,
                 bbox=dict(boxstyle='round', facecolor=bg, alpha=0.6))

plt.tight_layout()
plt.savefig('task3_results.png', dpi=100, bbox_inches='tight')
plt.show()

task3_acc = np.mean([r['correct'] for r in task3_results])
print(f"\n📊 Task 3 Accuracy on our 5 samples: {task3_acc:.1%}")
print(f"   Paper result for InternVL-2B on full dataset: 30.3%")
print(f"   Random guessing baseline (4 classes): 25%")
print(f"   A trained specialist AI: ~98% (from paper, Table 4)")

## Step 7: Error Analysis — Why Does the AI Fail?

The LMOD paper identified **6 failure modes** for LVLMs in ophthalmology. Let's see if we can spot any of them in our results!

| # | Failure Mode | What it means |
|---|-------------|---------------|
| E1 | **Misclassification** | The AI gets the answer wrong |
| E2 | **Failure to Abstain** | The AI confidently answers even when it shouldn't |
| E3 | **Inconsistent Reasoning** | The AI contradicts itself in the same response |
| E4 | **Hallucination** | The AI invents things that aren't in the image |
| E5 | **Assertion without Justification** | The AI gives an answer without explaining why |
| E6 | **Lack of Domain Knowledge** | The AI shows it doesn't know medical facts |

In [ ]:
# ─── Print full model responses for wrong predictions ─────────────────────────
print("=" * 70)
print("DETAILED ERROR ANALYSIS")
print("=" * 70)

for task_label, results, gt_key, pred_key in [
    ('TASK 2 (Glaucoma)',    task2_results, 'ground_truth', 'prediction'),
    ('TASK 3 (MH Staging)', task3_results, 'ground_truth', 'prediction'),
]:
    print(f"\n📋 {task_label} — Full Responses:")
    print("-" * 70)
    for i, r in enumerate(results):
        status = '✅ CORRECT' if r['correct'] else '❌ WRONG'
        gt_val   = r[gt_key]
        pred_val = r[pred_key]
        print(f"\nSample {i+1} [{r['sample']['id']}] — {status}")
        print(f"  Ground Truth: {gt_val}")
        print(f"  AI Prediction: {pred_val}")
        print("  Full response:")
        for line in r['response'].split('\n')[:6]:
            if line.strip():
                print(f"    {line.strip()}")

In [ ]:
# ─── Classify error types ────────────────────────────────────────────────────
def classify_error(response, ground_truth, prediction):
    """Rule-based error classification based on response text."""
    if prediction == ground_truth:
        return ['✅ Correct']
    r = response.lower()
    errors = []
    # E3: Inconsistent Reasoning — mentions multiple stages
    if isinstance(ground_truth, int):
        if sum(f'stage {i}' in r for i in range(1, 5)) > 1:
            errors.append('E3: Inconsistent Reasoning')
    # E5: Assertion without Justification — very short response
    if len(response.split()) < 20:
        errors.append('E5: Assertion without Justification')
    # E4: Hallucination — mentions clearly out-of-domain content
    if any(kw in r for kw in ['cat', 'dog', 'photograph of a person']):
        errors.append('E4: Hallucination')
    # E6: Lack of Domain Knowledge — invents stages that don't exist
    if isinstance(ground_truth, int) and 'stage 5' in r:
        errors.append('E6: Lack of Domain Knowledge (stage 5 does not exist!)')
    return errors or ['E1: Misclassification']


print("ERROR CLASSIFICATION\n" + "=" * 50)

for task_label, results, gt_key, pred_key in [
    ('Task 2 (Glaucoma)',    task2_results, 'ground_truth', 'prediction'),
    ('Task 3 (MH Staging)', task3_results, 'ground_truth', 'prediction'),
]:
    print(f"\n🔍 {task_label} Errors:")
    any_errors = False
    for r in results:
        if not r['correct']:
            errors = classify_error(r['response'], r[gt_key], r[pred_key])
            print(f"  [{r['sample']['id']}]: {' | '.join(errors)}")
            any_errors = True
    if not any_errors:
        print("  (All correct!  🎉)")

## Step 8: Results Summary — How Did Our AI Do?

Let's put everything together and compare with the paper's findings on the full dataset.

In [ ]:
# ─── Summary Dashboard ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Summary: InternVL3.5-8B on LMOD Tasks', fontsize=16, fontweight='bold')

our_results = {
    'Anatomical\nRecognition': np.mean([r['accuracy'] for r in task1_results]),
    'Glaucoma\nDiagnosis':     np.mean([r['correct']  for r in task2_results]),
    'Macular Hole\nStaging':   np.mean([r['correct']  for r in task3_results]),
}

paper_baselines = {
    'Anatomical\nRecognition': {'InternVL-2B\n(paper)': 0.4807, 'Random': 0.0,   'Expert NN': 0.98},
    'Glaucoma\nDiagnosis':     {'InternVL-2B\n(paper)': 0.50,   'Random': 0.50,  'Expert NN': 0.83},
    'Macular Hole\nStaging':   {'InternVL-2B\n(paper)': 0.303,  'Random': 0.25,  'Expert NN': 0.98},
}
demo_colors  = ['#3498db', '#2ecc71', '#e67e22']

for ax, (task_name, demo_color) in zip(axes, zip(paper_baselines.keys(), demo_colors)):
    our_acc   = our_results[task_name]
    baselines = paper_baselines[task_name]
    all_bars  = {'Our 3.5-8B': our_acc, **baselines}
    colors    = [demo_color, '#95a5a6', '#e74c3c', '#27ae60']
    bars = ax.bar(list(all_bars.keys()), list(all_bars.values()),
                  color=colors, edgecolor='black', linewidth=0.5, alpha=0.85)
    for bar, val in zip(bars, all_bars.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.set_title(task_name, fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel('Accuracy / F1 score')
    ax.set_xticklabels(list(all_bars.keys()), rotation=15, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('summary.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 65)
print("FINAL SUMMARY: AI Eye Doctor Performance")
print("=" * 65)

t1 = np.mean([r['accuracy'] for r in task1_results])
t2 = np.mean([r['correct']  for r in task2_results])
t3 = np.mean([r['correct']  for r in task3_results])

print(f"""
Task 1 — Anatomical Recognition (OCT):
  Our InternVL3.5-8B (5 samples):        {t1:.1%}
  Full dataset — InternVL-2B (paper):    48.1% F1
  Full dataset — Expert NN (paper):      98.4% F1

Task 2 — Glaucoma Diagnosis:
  Our InternVL3.5-8B (5 samples):        {t2:.1%}
  Full dataset — InternVL-2B (paper):    50.0%  (coin flip!)
  Full dataset — Expert NN (paper):      82.7%

Task 3 — Macular Hole Staging:
  Our InternVL3.5-8B (5 samples):        {t3:.1%}
  Full dataset — InternVL-2B (paper):    30.3%
  Random guessing (4 classes):           25.0%
  Full dataset — Expert NN (paper):      98.2%
""")

print("KEY TAKEAWAYS:")
print("  1. Even a powerful 8B model struggles with specialised medical imaging.")
print("  2. Glaucoma diagnosis is basically random guessing for ALL LVLMs")
print("     tested in the paper — a very hard task requiring expert knowledge.")
print("  3. Simple CNNs specifically trained on these tasks do FAR better.")
print("  4. The AI knows what an OCT scan IS — but fine-grained clinical")
print("     analysis is a completely different challenge.")
print()
print("  This is why the LMOD benchmark matters: it shows exactly WHERE AI")
print("  needs to improve before it can help doctors in real clinics!")

## Discussion: Big Questions to Think About 🤔

Now that we've seen AI in action, let's think critically:

---

### 1. Why does a "general" AI struggle with medical images?
InternVL2 was trained on billions of images from the internet. But OCT scans are rare, specialised images — the AI has seen very few of them compared to photos of cats, cars, or people. This is called the **domain gap** problem.

### 2. Is fine-tuning enough?
The paper tried to *fine-tune* (specialise) LLaVA-Med on OCT images — but it still failed, outputting gibberish like "Optic Optic Optic Optic...". This shows that specialising a large general model for a narrow medical task is *not* straightforward.

### 3. Should we trust AI for medical diagnosis?
Our results show the AI gets glaucoma diagnosis **no better than random chance**. What would happen if a doctor trusted the AI's output?

### 4. What would make the AI better?
- More medical imaging training data
- Ophthalmology-specific pre-training
- Better ways to incorporate clinical knowledge
- Uncertainty estimation — the AI saying "I'm not sure" instead of guessing

### 5. Is the supervised neural network "cheating"?
The specialist CNN/RETFound models achieve 98%+ — but they were **trained specifically on these tasks** with labelled examples. They can't have a conversation, can't generalise to new tasks, and can't explain their reasoning.

---

### 🌍 Why does this matter?
Over **2.2 billion people** worldwide have vision impairment, many in low-income regions without enough eye specialists. AI that can reliably diagnose eye diseases could save millions from going blind. The LMOD benchmark helps researchers measure and improve these systems.

---

### 🚀 Bonus Challenge
Try modifying the prompts above! For example:
- Can you give the model more context about macular hole stages and improve its accuracy?
- What happens if you ask the model to first describe what it sees, then give a diagnosis?
- Try few-shot prompting: show the model one example before asking about the test image.

In [ ]:
# ─── BONUS: Try an improved, more informative prompt ─────────────────────────
IMPROVED_MH_PROMPT = """\
This is an ophthalmology OCT (Optical Coherence Tomography) image showing a cross-section of the retina.

Macular holes are classified into 4 stages:
- Stage 1: A small pit (foveal detachment), the hole has not fully formed yet
- Stage 2: A small full-thickness hole (less than 400 micrometres)
- Stage 3: A large full-thickness hole (more than 400 micrometres), vitreous still attached
- Stage 4: A large full-thickness hole with complete vitreous detachment from the macula

Look carefully at the central region of the retina. Based on what you see, what is the stage of the macular hole?

Follow the format: Stage: <AN INTEGER between 1 and 4>; Justification: <EXPLANATION>."""

print("Running with IMPROVED prompt on Task 3...\n")
improved_correct = []

for i, sample in enumerate(oct_samples):
    img          = load_oct_image(sample, annotated=False)
    pixel_values = preprocess_image(img)
    prompt       = f'<image>\n{IMPROVED_MH_PROMPT}'

    with torch.no_grad():
        response = model.chat(tokenizer, pixel_values, prompt, generation_config)

    gt_stage   = sample['stage']
    pred_stage = parse_mh_stage_response(response)
    correct    = pred_stage == gt_stage
    improved_correct.append(correct)

    icon = '✅' if correct else '❌'
    print(f"Sample {i+1}: GT=Stage {gt_stage} | Improved AI=Stage {pred_stage} {icon}")
    print(f"  {response[:150]}...\n")

basic_acc    = np.mean([r['correct'] for r in task3_results])
improved_acc = np.mean(improved_correct)
delta = improved_acc - basic_acc
trend = '📈 Improved!' if delta > 0 else ('📉 Got worse' if delta < 0 else '➡️  Same result')

print(f"\n📊 Comparison:")
print(f"  Basic prompt accuracy:    {basic_acc:.1%}")
print(f"  Improved prompt accuracy: {improved_acc:.1%}  {trend}")
print("\n💡 Prompt engineering can sometimes help — but it's not a silver bullet!")

## 🎉 Congratulations — You've Run a Real AI Research Experiment!

### What you learned today:

1. **How LVLMs work** — AI models that combine vision and language to answer questions about images
2. **What ophthalmology AI benchmarks do** — They measure whether AI is good enough for real clinical tasks
3. **The gap between general and specialist AI** — General models struggle; specialist-trained models excel
4. **The 6 failure modes of medical AI** — Misclassification, hallucination, inconsistent reasoning, etc.
5. **Prompt engineering** — How the way you ask a question affects the AI's answer

### 🔗 Learn more
- **LMOD Paper**: [arxiv.org/abs/2410.01620](https://arxiv.org/abs/2410.01620)
- **InternVL**: [github.com/OpenGVLab/InternVL](https://github.com/OpenGVLab/InternVL)
- **LMOD Project Page**: [kfzyqin.github.io/lmod](https://kfzyqin.github.io/lmod/)

### 🤔 Questions to explore further
- Can you test GPT-4o with the OpenAI API and compare its results?
- Can you train a simple CNN on the REFUGE dataset and beat InternVL's glaucoma accuracy?
- What would an AI need to learn to close the gap with human experts?

---
*Based on: Qin et al., "LMOD: A Large Multimodal Ophthalmology Dataset and Benchmark for Large Vision-Language Models", arXiv:2410.01620*